# 03 — TextCNN Ponderado (deep learning, CPU-friendly)

Modelo compacto (Kim, 2014): embedding + convs 1D + max-pool + linear. Treinado do zero,
sempre sobre **`VOTO_LIMPO`**, com `CrossEntropyLoss(weight=...)` ponderada por frequência
inversa de classe.

**Saída:** `resultados/metricas_textcnn.json` e figura de matriz de confusão.


## 1. Setup


In [ ]:
import os, sys, subprocess
REPO_DIR = os.environ.get('REPO_DIR', '/content/deep-acordao-tcu2')
REPO_URL = 'https://github.com/bsousa7/deep-acordao-tcu2.git'
BRANCH = os.environ.get('BRANCH', 'claude/deep-acordao-tcu-refactor-yyjfr3')
if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
    subprocess.run(['git', 'clone', REPO_URL, '--branch', BRANCH, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
%pip -q install torch pandas pyarrow scikit-learn matplotlib


## 2. Carrega corpus


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, torch
torch.manual_seed(42); np.random.seed(42)

BASE = Path(REPO_DIR)

# Mesma lógica de persistência do 01/02: prefere o Google Drive se já tiver
# dados lá (permite rodar 03 numa sessão separada de 01/02).
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive') / 'deep-acordao-tcu2'
except Exception as _e:
    DRIVE_ROOT = None
    print(f'Google Drive indisponível (fora do Colab?): {_e}')

_drive_interim = (DRIVE_ROOT / 'data' / 'interim') if DRIVE_ROOT else None
PERSIST_BASE = DRIVE_ROOT if (_drive_interim and _drive_interim.exists()) else BASE

DATA_INTERIM = PERSIST_BASE / 'data' / 'interim'
RESULTADOS = PERSIST_BASE / 'resultados'
FIGURAS = RESULTADOS / 'figuras'; FIGURAS.mkdir(parents=True, exist_ok=True)

print(f'Lendo dados de: {PERSIST_BASE}')
df = pd.read_parquet(DATA_INTERIM / 'acordaos_rotulados.parquet')
print('corpus n =', len(df))
print(df['LABEL'].value_counts().to_string())


## 3. K-Fold 5× com pesos por frequência inversa


In [ ]:
from src.modelos.textcnn import treinar_kfold

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', DEVICE)

res_cnn = treinar_kfold(
    df, campo='VOTO_LIMPO', n_splits=5,
    max_vocab=20_000, max_len=300, epochs=8, device=DEVICE,
)
print(f"F1-macro = {res_cnn['f1_macro']:.4f}  |  acurácia = {res_cnn['accuracy']:.4f}")
print(res_cnn['classification_report'])


## 4. Matriz de confusão


In [ ]:
import matplotlib.pyplot as plt
cm = np.array(res_cnn['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5.2, 4.6))
im = ax.imshow(cm, cmap='Blues')
labels = ['Irregular', 'Reg. c/\nRessalva', 'Regular']
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
ax.set_xlabel('Previsto'); ax.set_ylabel('Verdadeiro')
ax.set_title('TextCNN — matriz de confusão (5-fold, VOTO_LIMPO)')
for i in range(3):
    for j in range(3):
        cor = 'white' if cm[i, j] > cm.max()/2 else 'black'
        ax.text(j, i, cm[i, j], ha='center', va='center', color=cor, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURAS / 'textcnn_matriz_confusao.png', dpi=150)
plt.show()


## 5. Persistência


In [ ]:
from src.avaliacao.metricas import salvar_json

saida = {
    'modelo': 'TextCNN (Kim 2014)',
    'feature': 'VOTO_LIMPO',
    'weighting': 'CrossEntropyLoss(weight=pesos_tensor(y_tr)) por fold',
    'kfold_5x': {
        'f1_macro': res_cnn['f1_macro'],
        'accuracy': res_cnn['accuracy'],
        'per_class_f1': res_cnn['per_class_f1'],
        'confusion_matrix': res_cnn['confusion_matrix'],
    },
    'hiperparametros': {
        'vocab_size': res_cnn['vocab_size'],
        'max_len': res_cnn['max_len'],
        'epochs': res_cnn['epochs'],
    },
}
salvar_json(saida, RESULTADOS / 'metricas_textcnn.json')
